# Pipeline Machine Learning — Détection de fraude assurance

Ce notebook présente clairement les étapes demandées par l'encadrant :

1. Exploration des données  
2. Transformation et nettoyage  
3. Analyse des variables influentes  
4. Lancement de l'entraînement avec séparation `X` / `y`

## 1. Exploration des données

Objectif : comprendre le dataset `insurance_claims.csv`, identifier les colonnes, les valeurs manquantes et la variable cible `fraud_reported`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT_DIR / "data" / "insurance_claims.csv"

df = pd.read_csv(DATA_PATH)

print("Taille du dataset :", df.shape)
print("\nColonnes :")
print(df.columns.tolist())

print("\nRépartition de la target fraud_reported :")
print(df["fraud_reported"].value_counts())

print("\nTaux de fraude :")
print(df["fraud_reported"].value_counts(normalize=True) * 100)

In [ ]:
df.head()

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
unknown_values = (df == "?").sum().sort_values(ascending=False)

print("Valeurs manquantes :")
print(missing_values[missing_values > 0])

print("\nValeurs '?' :")
print(unknown_values[unknown_values > 0])

## 2. Transformation et nettoyage

Les modèles Machine Learning ne travaillent pas directement avec les textes.  
Donc les variables catégorielles sont transformées en variables numériques avec `OneHotEncoder`.

La target est transformée comme suit :

- `Y` → `1` : fraude
- `N` → `0` : non fraude

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df_clean = df.replace("?", np.nan).copy()

columns_to_drop = [col for col in ["policy_number", "insured_zip", "_c39"] if col in df_clean.columns]

# X ne contient pas fraud_reported
X = df_clean.drop(columns=["fraud_reported"] + columns_to_drop)

# y contient uniquement la target
y = df_clean["fraud_reported"].map({"Y": 1, "N": 0})

print("Colonnes supprimées :", columns_to_drop)
print("Shape X :", X.shape)
print("Shape y :", y.shape)
print("Target y :")
print(y.value_counts())

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("\nVariables numériques :", len(numeric_features))
print("Variables catégorielles :", len(categorical_features))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape)
print("Test :", X_test.shape)

## 3. Analyse des variables influentes

Après l'entraînement, le projet génère un fichier `outputs/feature_importance.csv`.  
Il contient les variables qui influencent le plus la prédiction de la fraude.

In [ ]:
FEATURE_IMPORTANCE_PATH = ROOT_DIR / "outputs" / "feature_importance.csv"

if FEATURE_IMPORTANCE_PATH.exists():
    feature_importance = pd.read_csv(FEATURE_IMPORTANCE_PATH)
    display(feature_importance.head(20))
else:
    print("Le fichier feature_importance.csv n'existe pas encore. Lancez d'abord : python train_model.py")

## 4. Lancement de l'entraînement

Dans cette étape, on entraîne plusieurs modèles de classification.  
La colonne `fraud_reported` est isolée dans `y` et n'est pas incluse dans `X`.

Cela évite de donner la réponse directement au modèle.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "ExtraTreesClassifier": ExtraTreesClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
}

results = []

for name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    y_proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf.named_steps["model"], "predict_proba") else None
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_fraud": precision_score(y_test, y_pred, zero_division=0),
        "recall_fraud": recall_score(y_test, y_pred, zero_division=0),
        "f1_fraud": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc,
    })

results_df = pd.DataFrame(results).sort_values(
    by=["f1_fraud", "recall_fraud", "roc_auc"],
    ascending=False
)

results_df

## Conclusion

Le pipeline suivi est :

1. Comprendre et explorer les données  
2. Nettoyer les valeurs manquantes et transformer les textes en nombres  
3. Isoler la target `fraud_reported` dans `y`  
4. Utiliser les autres colonnes dans `X`  
5. Entraîner plusieurs modèles  
6. Choisir le modèle le plus adapté  
7. Générer une probabilité de fraude et un niveau de risque pour chaque dossier